In [1]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import Trainer, TrainingArguments
import json, random
import torch
import torch.nn as nn
from datasets import Dataset
from peft import LoraConfig, TaskType, get_peft_model
from peft import LoraConfig, get_peft_model
import os
from transformers import Qwen2ForSequenceClassification, Qwen2Tokenizer
import torch
import json
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import Trainer, TrainingArguments
import json, random
import torch
import torch.nn as nn
import numpy as np
from datasets import Dataset
from peft import PeftModel


In [4]:
# 一些参数
model_path = '/root/autodl-tmp/self-llm/model/Qwen/Qwen2___5-1___5B-Instruct'
train_data_path = '../../dataset/prep/2/train.json'
test_file_path = '../../dataset/prep/2/test.json'

model_path = '/root/autodl-tmp/self-llm/model/Qwen/Qwen2___5-1___5B-Instruct'
train_data_path = '../../dataset/prep/2/train.json'
test_file_path = '../../dataset/prep/2/test.json'
res_path = '../../dataset/res/tmp_res2.json'
save_path = "./output/save_model_7b/"

# 加载预训练的 Qwen2 模型和分词器
tokenizer = AutoTokenizer.from_pretrained(model_path)

# 创建标签到索引的映射
label_to_id = {
    "气虚血瘀证": 0,
    "痰瘀互结证": 1,
    "气阴两虚证": 2,
    "气滞血瘀证": 3,
    "肝阳上亢证": 4,
    "阴虚阳亢证": 5,
    "痰热蕴结证": 6,
    "痰湿痹阻证": 7,
    "阳虚水停证": 8,
    "肝肾阴虚证": 9,
}

In [11]:
# 读取jsonl文件
with open(train_data_path, 'r', encoding='utf-8') as file:
    # 使用 json.load() 方法将文件内容解析为 Python 对象
    data = json.load(file)
random.shuffle(data)



num_labels = len(label_to_id)  # 根据你的标签数量设置num_labels


# 将文本标签转换为数值标签
label_list = []
for num, example in enumerate(data):
    labels = [0] * num_labels  
    label_num = [label_to_id[i] for i in example['output'].split('|')]
    for i in label_num:  # 遍历每个标签
        labels[i] = 1  # 将对应位置设置为 1
    label_list.append(labels)


In [14]:

# 将数据转换为datasets库的Dataset对象
dataset = Dataset.from_list(data)

# 将数据集拆分为训练集和验证集
dataset = dataset.train_test_split(test_size=0.2)

# 定义一个函数来处理数据集中的文本
def preprocess_function(examples, indices=None):
     # 对文本进行分词
    encoding = tokenizer(
        examples["input"], truncation=True, padding="max_length", max_length=128
    )
    
    # 根据当前批次的索引生成对应的 labels
    encoding["label"] = [label_list[i] for i in indices]  # 获取当前批次的标签
    # encoding["labels"] = torch.tensor(batch_labels, dtype=torch.float32)
    return encoding

# 对数据集进行预处理
encoded_dataset = dataset.map(preprocess_function, batched=True, with_indices=True)

# 打印训练集和验证集中的一些样本
print("Train dataset sample:")
print(encoded_dataset['train'][0])  # 打印训练集中的第一个样本

print("Eval dataset sample:")
print(encoded_dataset['test'][0])  # 打印验证集中的第一个样本

Map:   0%|          | 0/640 [00:00<?, ? examples/s]

Map:   0%|          | 0/160 [00:00<?, ? examples/s]

Train dataset sample:
{'instruction': '\n任务：根据患者[基本信息],[主诉],[症状],[中医望闻切诊],[病史],[体格检查],[辅助检查]等信息,输出患者证型类别\n', 'input': '\n[症状]:发作性心慌，无胸闷，无心前区疼痛，无头晕头痛，无咳嗽咳痰，偶有反酸烧心，无一过性黑蒙，无晕厥，纳眠可，二便调。\n[中医望闻切诊]:中医望闻切诊：表情自然，面色红润，形体正常，望态，语气清，气息平，无异常气味，舌淡白、苔薄黄,脉弦数。\n', 'output': '气滞血瘀证', 'input_ids': [198, 58, 101368, 5669, 107253, 33071, 63109, 102838, 3837, 42192, 100277, 102706, 3837, 42192, 63109, 24562, 23836, 105748, 3837, 42192, 116280, 109180, 3837, 42192, 109244, 103298, 110776, 3837, 100583, 18830, 94443, 99918, 100228, 63109, 3837, 42192, 14777, 38182, 33071, 56652, 100750, 3837, 42192, 102997, 118991, 3837, 99458, 101519, 30440, 3837, 40820, 99364, 47872, 8997, 58, 104823, 99317, 99608, 99322, 99781, 5669, 104823, 99317, 99608, 99322, 99781, 5122, 102936, 99795, 3837, 113906, 99425, 99842, 3837, 82699, 31914, 100416, 3837, 99317, 35243, 3837, 110098, 79766, 3837, 103007, 49111, 3837, 42192, 70633, 111191, 3837, 101601, 99773, 99243, 5373, 114144, 101264, 99789, 11, 100297, 106514, 8863, 8997, 15

In [20]:

class MultiLabelModel(nn.Module):
    def __init__(self, model_name, num_labels):
        super(MultiLabelModel, self).__init__()
        self.qwen = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=num_labels)
        self.qwen.config.pad_token_id = 151643  # 定义pad token，模型才会忽略后面那些pad而是把真正最后一个token的hidden state用于分类
        # self.loss_fn = nn.BCEWithLogitsLoss(pos_weight=class_weights)  # 多标签分类损失函数
        self.loss_fn = nn.BCEWithLogitsLoss()  # 多标签分类损失函数


    def forward(self, input_ids, attention_mask, labels=None):
        outputs = self.qwen(input_ids=input_ids, attention_mask=attention_mask)
        logits = outputs.logits
        if labels is not None:
            loss = self.loss_fn(logits, labels.float())
            return {"loss": loss, "logits": logits}
        return {"logits": logits}
    def save_pretrained(self, save_directory):
        """保存模型的权重和配置文件"""
        # 保存模型的权重
        self.qwen.save_pretrained(save_directory)
        # 保存模型的配置
        self.qwen.config.save_pretrained(save_directory)
    
model = MultiLabelModel(model_path, num_labels=num_labels)  
model = get_peft_model(model, config)
model.print_trainable_parameters()
# print([(n, type(m)) for n, m in model.named_modules()])
# print(model)






Some weights of Qwen2ForSequenceClassification were not initialized from the model checkpoint at /root/autodl-tmp/self-llm/model/Qwen/Qwen2___5-1___5B-Instruct and are newly initialized: ['score.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


trainable params: 4,622,376 || all params: 1,548,352,040 || trainable%: 0.298535209085913


In [21]:

config = LoraConfig(
    r=4,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj",'score'],
    lora_dropout=0.1,
)

num_train_epochs=2
per_device_train_batch_size=2
per_device_eval_batch_size=2
warmup_steps=50
weight_decay=0.01
logging_steps=1
use_cpu=False

In [22]:



# 定义训练参数
training_args = TrainingArguments(
    output_dir=save_path,                           # 输出目录
    num_train_epochs=num_train_epochs,              # 训练的epoch数
    per_device_train_batch_size=per_device_train_batch_size,    # 每个设备的训练batch size
    per_device_eval_batch_size=per_device_eval_batch_size,      # 每个设备的评估batch size
    warmup_steps=warmup_steps,                  # 预热步数
    weight_decay=weight_decay,                  # 权重衰减
    logging_dir=save_path,                      # 日志目录
    logging_steps=logging_steps,
    evaluation_strategy="epoch",
    save_strategy="epoch",    # 每个epoch保存一次检查点
    save_total_limit=3,       # 最多保存3个检查点，旧的会被删除
    use_cpu=False
)

from transformers import Trainer
class CustomTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        # print(f"Inputs keys: {inputs.keys()}") 
        #  # 调试：打印输入数据，查看输入格式
        # try:
        #     print(f"Inputs to the model: {inputs}")
        # except Exception as e:
        #     print(f"Error when printing inputs: {e}")
        
        # # 确保包含 labels
        # if "labels" not in inputs:
        #     raise ValueError("The 'label' key is missing in the inputs!")

        labels = inputs["labels"]  # 提取标签
        labels = labels.float()  # 确保标签是 float 类型
        # print(labels.shape)
        inputs_ids = inputs['input_ids']
        attention_mask = inputs['attention_mask']
        outputs = model(inputs_ids, attention_mask, labels)  # 模型的输出
        logits = outputs["logits"]  # 提取 logits
        # print(logits.shape)# 计算损失
        loss = outputs['loss']
        return (loss, outputs) if return_outputs else loss






# 定义Trainer
trainer = CustomTrainer(
    model=model,                                    # 模型
    args=training_args,                             # 训练参数
    train_dataset=encoded_dataset['train'],         # 训练数据集
    eval_dataset=encoded_dataset['test']            # 评估数据集
)

/root/miniconda3/lib/python3.10/site-packages/transformers/training_args.py:1525: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


In [23]:
# 开始训练
trainer.train()
trainer.save_state()
model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)

Epoch,Training Loss,Validation Loss
1,0.425800,No log
2,0.327700,No log


We detected that you are passing `past_key_values` as a tuple and this is deprecated and will be removed in v4.43. Please use an appropriate `Cache` class (https://huggingface.co/docs/transformers/v4.41.3/en/internal/generation_utils#transformers.Cache)


('./output/save_model_7b/tokenizer_config.json',
 './output/save_model_7b/special_tokens_map.json',
 './output/save_model_7b/vocab.json',
 './output/save_model_7b/merges.txt',
 './output/save_model_7b/added_tokens.json',
 './output/save_model_7b/tokenizer.json')

In [5]:



id_to_label = {v: k for k, v in label_to_id.items()}

num_labels = len(label_to_id)  # 根据你的标签数量设置num_labels

# 加载预训练的 Qwen2 模型和分词器
tokenizer = AutoTokenizer.from_pretrained(model_path)

class MultiLabelModel(nn.Module):
    def __init__(self, model_name, num_labels):
        super(MultiLabelModel, self).__init__()
        self.qwen = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=num_labels)
        self.qwen.config.pad_token_id = 151643  # 定义pad token，模型才会忽略后面那些pad而是把真正最后一个token的hidden state用于分类
        self.loss_fn = nn.BCEWithLogitsLoss()  # 多标签分类损失函数

    def forward(self, input_ids, attention_mask, labels=None):
        outputs = self.qwen(input_ids=input_ids, attention_mask=attention_mask)
        logits = outputs.logits
        if labels is not None:
            loss = self.loss_fn(logits, labels.float())
            return {"loss": loss, "logits": logits}
        return {"logits": logits}
    
model = MultiLabelModel(model_path, num_labels=num_labels)
#合并lora
model = PeftModel.from_pretrained(model, model_id=save_path)
for parameter in model.parameters():
    parameter.requires_grad = False


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

# 准备输入文本
texts = []
true_label = []
ID = []

with open(test_file_path,'r',encoding='utf-8') as file:
    data = json.load(file)
    for line in data:
        t = line['input']
        texts.append(t)
        labels = [0] * num_labels  # 初始化多热编码向量（全 0）
        # label_num = [label_to_id[i] for i in line['output'].split('|')]
        # for i in label_num:  # 遍历每个标签
        #     labels[i] = 1  # 将对应位置设置为 1
        # true_label.append(labels)
        ID.append(line['ID'])


# 对文本进行编码
inputs = tokenizer(texts, padding=True, truncation=True, return_tensors="pt")
inputs = {k: v.to(device) for k, v in inputs.items()}

# 进行推理
with torch.no_grad():
    outputs = model(**inputs)

# 获取预测结果
logits = outputs['logits']
predictions = torch.sigmoid(logits)
# threshold = find_threshold_micro(logits.cpu().detach().numpy(), torch.tensor(true_label).cpu().detach().numpy())
# for i in predictions:
#     print(i)
# 使用 torch.topk 获取每行的前两大值
topk_values, topk_indices = torch.topk(predictions, 2, dim=1)
# 获取最大值和次大值
top_max_values = topk_values[:, 0]  # 第一大值
second_max_values = topk_values[:, 1]  # 第二大值
# 对于每个 batch，检查最大值和次大值的差值
diff = top_max_values - second_max_values
# 如果差值大于0.2，则将次大值设置为最大值
second_max_values[diff > 0.1] = top_max_values[diff > 0.1]
predictions = torch.eq(predictions, second_max_values.unsqueeze(1))

# 检查对应位置是否相同
# 使用 np.all 逐行比较证型的正确率
yhat_raw_syndrome = predictions.to(torch.int).cpu().detach().numpy()
# y_syndrome = torch.tensor(true_label).cpu().detach().numpy()
# comparison_syndrome = np.all(yhat_raw_syndrome == y_syndrome, axis=1)
# matching_rows_count_syndrome = np.sum(comparison_syndrome)
# ACC_syndrome = matching_rows_count_syndrome / yhat_raw_syndrome.shape[0]
# print('ACC:{}'.format(ACC_syndrome))

# 写入json文件中
results = []
yhat_raw_syndrome = yhat_raw_syndrome.tolist()
for num, i in enumerate(yhat_raw_syndrome):
    result = {}
    # 使用 enumerate 查找值为 1 的位置
    positions = [index for index, value in enumerate(i) if value == 1]
    # 将位置转换为标签
    labels = [id_to_label[position] for position in positions]
    result['ID'] = ID[num]
    result['证型'] = '|'.join(labels)
    results.append(result)



Some weights of Qwen2ForSequenceClassification were not initialized from the model checkpoint at /root/autodl-tmp/self-llm/model/Qwen/Qwen2___5-1___5B-Instruct and are newly initialized: ['score.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
We detected that you are passing `past_key_values` as a tuple and this is deprecated and will be removed in v4.43. Please use an appropriate `Cache` class (https://huggingface.co/docs/transformers/v4.41.3/en/internal/generation_utils#transformers.Cache)


In [6]:
with open(res_path, 'w', encoding='utf-8') as f:
    json.dump(results, f, ensure_ascii=False, indent=4)